# Preparación dataset de animales (perro/gato/pájaro)

Este notebook:
1. Descarga datasets de Kaggle de perros/gatos y pájaros.
2. Extrae las imágenes.
3. Balancea las 3 clases (mismo número de imágenes por clase).
4. Separa en train/val/test con estructura:

../data/animales/
  train/perro, gato, pajaro
  val/perro,   gato, pajaro
  test/perro,  gato, pajaro

Este dataset será usado en el notebook `01_cnn_animales.ipynb` para entrenar las CNN.


In [21]:
# Celda 2: Imports, rutas y configuración general

import os
from pathlib import Path
import random
import shutil

import numpy as np

# La librería kaggle se utiliza para acceder a la API oficial de Kaggle desde Python.
# Si no está instalada, se indicará un mensaje al usuario.
try:
    from kaggle.api.kaggle_api_extended import KaggleApi
except ImportError:
    print("La librería 'kaggle' no está instalada. Se recomienda ejecutar:")
    print("!pip install kaggle")

# Fijación de semillas para garantizar reproducibilidad en las particiones train/val/test.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Se asume que este notebook se encuentra en: proyecto_final_aa/notebooks/
BASE_DIR = Path("..").resolve()

# Carpeta donde se almacenarán los archivos descargados de Kaggle.
RAW_DIR = BASE_DIR / "data" / "raw_kaggle"

# Carpeta final donde se dejará el dataset preparado para los modelos.
FINAL_DATA_DIR = BASE_DIR / "data" / "animales"

print("BASE_DIR       :", BASE_DIR)
print("RAW_DIR        :", RAW_DIR)
print("FINAL_DATA_DIR :", FINAL_DATA_DIR)


BASE_DIR       : C:\Users\DANIELA\Documents\proyecto_final_procesamiento
RAW_DIR        : C:\Users\DANIELA\Documents\proyecto_final_procesamiento\data\raw_kaggle
FINAL_DATA_DIR : C:\Users\DANIELA\Documents\proyecto_final_procesamiento\data\animales


# Preparacion de Dataset



In [22]:
# Celda 3: Inicialización de la API de Kaggle

# Si la librería kaggle no estaba instalada, el siguiente import fallará.
# En ese caso, es necesario instalarla previamente con: !pip install kaggle
from kaggle.api.kaggle_api_extended import KaggleApi

# Se asegura la existencia de la carpeta donde se guardarán los datos sin procesar.
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Se crea una instancia de la API y se autentica utilizando el archivo kaggle.json.
api = KaggleApi()
api.authenticate()

print("Autenticación en Kaggle completada correctamente.")


Autenticación en Kaggle completada correctamente.


In [23]:
# Celda 4: Descarga de los datasets de Kaggle (gatos/perros y pájaros)

# Identificadores de los datasets en Kaggle:
# - tongpython/cat-and-dog → imágenes de gatos y perros
# - veeralakrishna/200-bird-species-with-11788-images → imágenes de múltiples especies de pájaros
CAT_DOG_DATASET = "tongpython/cat-and-dog"
BIRDS_DATASET   = "veeralakrishna/200-bird-species-with-11788-images"

# Directorios donde se descomprimirán los contenidos de cada dataset.
cat_dog_dir = RAW_DIR / "cat_dog"
birds_dir   = RAW_DIR / "birds_200"

cat_dog_dir.mkdir(parents=True, exist_ok=True)
birds_dir.mkdir(parents=True, exist_ok=True)

print("Descargando dataset de gatos y perros...")
api.dataset_download_files(
    CAT_DOG_DATASET,
    path=cat_dog_dir,
    unzip=True
)
print("Descarga de gatos/perros finalizada.")

print("\nDescargando dataset de pájaros (200 especies)...")
api.dataset_download_files(
    BIRDS_DATASET,
    path=birds_dir,
    unzip=True
)
print("Descarga de pájaros finalizada.")


Descargando dataset de gatos y perros...
Dataset URL: https://www.kaggle.com/datasets/tongpython/cat-and-dog
Descarga de gatos/perros finalizada.

Descargando dataset de pájaros (200 especies)...
Dataset URL: https://www.kaggle.com/datasets/veeralakrishna/200-bird-species-with-11788-images
Descarga de pájaros finalizada.


In [24]:
# Celda 5: Exploración mínima de las carpetas descargadas

def list_some_paths(root: Path, max_items: int = 10):
    """
    Muestra algunas rutas internas bajo la carpeta 'root' para inspección rápida.
    Esta función es útil para entender cómo viene organizado el dataset original.
    """
    print(f"\nListado parcial de contenidos en: {root}")
    count = 0
    for p in root.rglob("*"):
        print(" -", p.relative_to(root))
        count += 1
        if count >= max_items:
            break

print("=== Estructura del dataset de perros/gatos (cat_dog) ===")
list_some_paths(cat_dog_dir)

print("\n=== Estructura del dataset de pájaros (birds_200) ===")
list_some_paths(birds_dir)


=== Estructura del dataset de perros/gatos (cat_dog) ===

Listado parcial de contenidos en: C:\Users\DANIELA\Documents\proyecto_final_procesamiento\data\raw_kaggle\cat_dog
 - test_set
 - training_set
 - test_set\test_set
 - training_set\training_set
 - training_set\training_set\cats
 - training_set\training_set\dogs
 - training_set\training_set\cats\cat.1.jpg
 - training_set\training_set\cats\cat.10.jpg
 - training_set\training_set\cats\cat.100.jpg
 - training_set\training_set\cats\cat.1000.jpg

=== Estructura del dataset de pájaros (birds_200) ===

Listado parcial de contenidos en: C:\Users\DANIELA\Documents\proyecto_final_procesamiento\data\raw_kaggle\birds_200
 - CUB_200_2011.tgz
 - segmentations.tgz


In [38]:
# Celda 5: Extracción del archivo CUB_200_2011.tgz (dataset de pájaros)

import tarfile

tgz_path = birds_dir / "CUB_200_2011.tgz"

if tgz_path.exists():
    print("Se encontró el archivo:", tgz_path)
    # Esta extracción puede tardar algunos segundos.
    with tarfile.open(tgz_path, mode="r:gz") as tar:
        tar.extractall(path=birds_dir)
    print("Extracción completada.")

    # Se muestra la carpeta principal creada tras la extracción.
    extracted_root = birds_dir / "CUB_200_2011"
    print("Carpeta principal del dataset de pájaros:", extracted_root)
else:
    print("No se encontró CUB_200_2011.tgz en", birds_dir)


Se encontró el archivo: C:\Users\DANIELA\Documents\proyecto_final_procesamiento\data\raw_kaggle\birds_200\CUB_200_2011.tgz


C:\Users\DANIELA\AppData\Local\Temp\ipykernel_3936\357961490.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=birds_dir)


Extracción completada.
Carpeta principal del dataset de pájaros: C:\Users\DANIELA\Documents\proyecto_final_procesamiento\data\raw_kaggle\birds_200\CUB_200_2011


In [39]:
# Celda 6: Definición de la carpeta raíz de imágenes de pájaros

BIRDS_ROOT = birds_dir / "CUB_200_2011" / "images"

print("BIRDS_ROOT:", BIRDS_ROOT)
print("¿Existe BIRDS_ROOT?:", BIRDS_ROOT.exists())


BIRDS_ROOT: C:\Users\DANIELA\Documents\proyecto_final_procesamiento\data\raw_kaggle\birds_200\CUB_200_2011\images
¿Existe BIRDS_ROOT?: True


In [40]:
# Celda 6: Localización de carpetas de imágenes de gatos y perros

from typing import List

# Extensiones de imagen consideradas para el procesamiento.
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp"]

def has_images(folder: Path) -> bool:
    """
    Verifica si una carpeta contiene al menos una imagen con alguna
    de las extensiones definidas en IMAGE_EXTENSIONS.
    """
    for ext in IMAGE_EXTENSIONS:
        if any(folder.glob(f"*{ext}")):
            return True
    return False

def find_class_dirs(root: Path, keywords: List[str]) -> List[Path]:
    """
    Busca, dentro de 'root', subcarpetas cuyo nombre contenga alguna
    de las palabras clave proporcionadas y que, además, contengan imágenes.
    
    Parámetros:
        root: carpeta raíz donde se realiza la búsqueda.
        keywords: lista de palabras clave (ejemplo: ["cat", "cats"]).
    
    Retorna:
        Lista de rutas Path que representan carpetas de clase válidas.
    """
    result = []
    for p in root.rglob("*"):
        if p.is_dir():
            name_lower = p.name.lower()
            if any(k in name_lower for k in keywords) and has_images(p):
                result.append(p)
    return result

# Búsqueda de carpetas de gatos y perros dentro del dataset descargado.
cat_dirs = find_class_dirs(cat_dog_dir, ["cat", "cats"])
dog_dirs = find_class_dirs(cat_dog_dir, ["dog", "dogs"])

print("Carpetas detectadas para CATS:")
for d in cat_dirs:
    print(" -", d)

print("\nCarpetas detectadas para DOGS:")
for d in dog_dirs:
    print(" -", d)


Carpetas detectadas para CATS:
 - C:\Users\DANIELA\Documents\proyecto_final_procesamiento\data\raw_kaggle\cat_dog\training_set\training_set\cats
 - C:\Users\DANIELA\Documents\proyecto_final_procesamiento\data\raw_kaggle\cat_dog\test_set\test_set\cats

Carpetas detectadas para DOGS:
 - C:\Users\DANIELA\Documents\proyecto_final_procesamiento\data\raw_kaggle\cat_dog\training_set\training_set\dogs
 - C:\Users\DANIELA\Documents\proyecto_final_procesamiento\data\raw_kaggle\cat_dog\test_set\test_set\dogs


In [47]:
# Celda 7 (versión corregida): Construcción de listas de imágenes por clase

def gather_images_from_dirs_case_insensitive(dirs: List[Path]) -> List[Path]:
    """
    Reúne todas las imágenes contenidas dentro de una lista de carpetas,
    teniendo en cuenta extensiones en cualquier combinación de mayúsculas/minúsculas.
    """
    all_imgs = []
    for d in dirs:
        for p in d.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:
                all_imgs.append(p)
    return all_imgs

# Imágenes de perros y gatos a partir del dataset cat_dog.
dog_images = gather_images_from_dirs_case_insensitive(dog_dirs)
cat_images = gather_images_from_dirs_case_insensitive(cat_dirs)

# Imágenes de pájaros: se utilizan todas las imágenes encontradas en el dataset birds_200.
bird_images = []
for p in birds_dir.rglob("*"):
    # Se revisa cada archivo bajo birds_dir y se filtra por extensión, ignorando mayúsculas/minúsculas.
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:
        bird_images.append(p)

print(f"Número de imágenes de perro : {len(dog_images)}")
print(f"Número de imágenes de gato  : {len(cat_images)}")
print(f"Número de imágenes de pájaro: {len(bird_images)}")

# Se muestran algunas rutas de ejemplo para verificar que realmente se encontraron imágenes de pájaros.
print("\nEjemplos de imágenes de pájaros:")
for p in bird_images[:10]:
    print(" -", p)


Número de imágenes de perro : 5017
Número de imágenes de gato  : 5011
Número de imágenes de pájaro: 11788

Ejemplos de imágenes de pájaros:
 - C:\Users\DANIELA\Documents\proyecto_final_procesamiento\data\raw_kaggle\birds_200\CUB_200_2011\images\001.Black_footed_Albatross\Black_Footed_Albatross_0001_796111.jpg
 - C:\Users\DANIELA\Documents\proyecto_final_procesamiento\data\raw_kaggle\birds_200\CUB_200_2011\images\001.Black_footed_Albatross\Black_Footed_Albatross_0002_55.jpg
 - C:\Users\DANIELA\Documents\proyecto_final_procesamiento\data\raw_kaggle\birds_200\CUB_200_2011\images\001.Black_footed_Albatross\Black_Footed_Albatross_0003_796136.jpg
 - C:\Users\DANIELA\Documents\proyecto_final_procesamiento\data\raw_kaggle\birds_200\CUB_200_2011\images\001.Black_footed_Albatross\Black_Footed_Albatross_0005_796090.jpg
 - C:\Users\DANIELA\Documents\proyecto_final_procesamiento\data\raw_kaggle\birds_200\CUB_200_2011\images\001.Black_footed_Albatross\Black_Footed_Albatross_0006_796065.jpg
 - C:\Use

In [41]:
# Celda 7: Construcción de listas de imágenes por clase (perro, gato, pájaro)

def gather_images_from_dirs(dirs: List[Path]) -> List[Path]:
    """
    Reúne todas las imágenes contenidas dentro de una lista de carpetas.
    
    Parámetros:
        dirs: lista de carpetas (Path) desde las cuales se extraerán imágenes.
    
    Retorna:
        Lista de rutas Path a cada imagen encontrada.
    """
    all_imgs = []
    for d in dirs:
        for ext in IMAGE_EXTENSIONS:
            all_imgs.extend(d.rglob(f"*{ext}"))
    return all_imgs

# Imágenes de perros y gatos a partir del dataset cat_dog.
dog_images = gather_images_from_dirs(dog_dirs)
cat_images = gather_images_from_dirs(cat_dirs)

# Imágenes de pájaros: se utilizan todas las imágenes encontradas en el dataset birds_200.
bird_images = []
for ext in IMAGE_EXTENSIONS:
    bird_images.extend(birds_dir.rglob(f"*{ext}"))

print(f"Número de imágenes de perro : {len(dog_images)}")
print(f"Número de imágenes de gato  : {len(cat_images)}")
print(f"Número de imágenes de pájaro: {len(bird_images)}")


Número de imágenes de perro : 5017
Número de imágenes de gato  : 5011
Número de imágenes de pájaro: 11788


In [42]:
# Celda 8: Balanceo de las tres clases (mismo número de imágenes)

# Se determina el número mínimo de imágenes disponible entre perro, gato y pájaro.
min_count = min(len(dog_images), len(cat_images), len(bird_images))
print("Cantidad mínima común de imágenes por clase:", min_count)

# Parámetro opcional para limitar manualmente el máximo por clase (por ejemplo, para acelerar pruebas).
MAX_PER_CLASS = None  # Si se desea limitar, se puede cambiar a un entero, por ejemplo 3000.

if MAX_PER_CLASS is not None:
    min_count = min(min_count, MAX_PER_CLASS)
    print("Se utiliza un máximo de imágenes por clase de:", MAX_PER_CLASS)

def sample_and_shuffle(img_list, n, seed=SEED):
    """
    Selecciona 'n' imágenes de forma aleatoria y reproducible desde 'img_list'.
    
    Parámetros:
        img_list: lista original de rutas de imágenes.
        n: cantidad de imágenes a seleccionar.
        seed: semilla utilizada para el generador aleatorio.
    
    Retorna:
        Lista de rutas seleccionadas, mezcladas de forma determinista.
    """
    img_list = img_list.copy()
    random.Random(seed).shuffle(img_list)
    return img_list[:n]

# Selección balanceada de imágenes para cada clase.
dog_balanced  = sample_and_shuffle(dog_images,  min_count, seed=SEED)
cat_balanced  = sample_and_shuffle(cat_images,  min_count, seed=SEED + 1)
bird_balanced = sample_and_shuffle(bird_images, min_count, seed=SEED + 2)

print(f"Imágenes de perro seleccionadas   : {len(dog_balanced)}")
print(f"Imágenes de gato seleccionadas    : {len(cat_balanced)}")
print(f"Imágenes de pájaro seleccionadas  : {len(bird_balanced)}")


Cantidad mínima común de imágenes por clase: 5011
Imágenes de perro seleccionadas   : 5011
Imágenes de gato seleccionadas    : 5011
Imágenes de pájaro seleccionadas  : 5011


In [43]:
# Celda 9: Función para dividir las listas en conjuntos train/val/test

def split_train_val_test(paths, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, seed=SEED):
    """
    Divide una lista de rutas en tres subconjuntos: train, val y test.
    
    Parámetros:
        paths: lista de rutas de imágenes.
        train_ratio: proporción para el conjunto de entrenamiento.
        val_ratio: proporción para el conjunto de validación.
        test_ratio: proporción para el conjunto de prueba.
        seed: semilla para mezclar la lista de forma reproducible.
    
    Retorna:
        (train_paths, val_paths, test_paths): tupla con las listas repartidas.
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Los ratios deben sumar 1.0"
    
    paths = paths.copy()
    random.Random(seed).shuffle(paths)
    
    n = len(paths)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)
    
    train_paths = paths[:n_train]
    val_paths   = paths[n_train:n_train + n_val]
    test_paths  = paths[n_train + n_val:]
    
    return train_paths, val_paths, test_paths

# División de cada clase en train, val y test.
train_dog, val_dog, test_dog = split_train_val_test(dog_balanced,  seed=SEED)
train_cat, val_cat, test_cat = split_train_val_test(cat_balanced,  seed=SEED + 10)
train_bir, val_bir, test_bir = split_train_val_test(bird_balanced, seed=SEED + 20)

print("Perro  -> train:", len(train_dog), " val:", len(val_dog), " test:", len(test_dog))
print("Gato   -> train:", len(train_cat), " val:", len(val_cat), " test:", len(test_cat))
print("Pájaro -> train:", len(train_bir), " val:", len(val_bir), " test:", len(test_bir))


Perro  -> train: 3507  val: 751  test: 753
Gato   -> train: 3507  val: 751  test: 753
Pájaro -> train: 3507  val: 751  test: 753


In [44]:
# Celda 10: Creación de la estructura ../data/animales/train|val|test/perro|gato|pajaro

splits = ["train", "val", "test"]
classes = ["perro", "gato", "pajaro"]

for split in splits:
    for cls in classes:
        folder = FINAL_DATA_DIR / split / cls
        folder.mkdir(parents=True, exist_ok=True)

print("Estructura de carpetas creada en:", FINAL_DATA_DIR)


Estructura de carpetas creada en: C:\Users\DANIELA\Documents\proyecto_final_procesamiento\data\animales


In [45]:
# Celda 11: Copiado de imágenes a la estructura final con nombres estándar

def copy_images(files, dest_dir: Path, prefix: str):
    """
    Copia una lista de imágenes a un directorio de destino asignando nombres nuevos
    con un prefijo dado y un índice incremental.
    
    Parámetros:
        files: lista de rutas de imágenes a copiar.
        dest_dir: ruta de la carpeta destino.
        prefix: prefijo que se utilizará en el nombre de los archivos de salida.
    """
    dest_dir.mkdir(parents=True, exist_ok=True)
    for i, src in enumerate(files, start=1):
        ext = src.suffix.lower()
        if ext not in IMAGE_EXTENSIONS:
            # Si la extensión no está en la lista, se fuerza a .jpg para homogeneidad.
            ext = ".jpg"
        new_name = f"{prefix}_{i:05d}{ext}"
        dest_path = dest_dir / new_name
        shutil.copy2(src, dest_path)

# Copiado de imágenes de PERROS.
copy_images(train_dog, FINAL_DATA_DIR / "train" / "perro",  "perro_train")
copy_images(val_dog,   FINAL_DATA_DIR / "val"   / "perro",  "perro_val")
copy_images(test_dog,  FINAL_DATA_DIR / "test"  / "perro",  "perro_test")

# Copiado de imágenes de GATOS.
copy_images(train_cat, FINAL_DATA_DIR / "train" / "gato",   "gato_train")
copy_images(val_cat,   FINAL_DATA_DIR / "val"   / "gato",   "gato_val")
copy_images(test_cat,  FINAL_DATA_DIR / "test"  / "gato",   "gato_test")

# Copiado de imágenes de PÁJAROS.
copy_images(train_bir, FINAL_DATA_DIR / "train" / "pajaro", "pajaro_train")
copy_images(val_bir,   FINAL_DATA_DIR / "val"   / "pajaro", "pajaro_val")
copy_images(test_bir,  FINAL_DATA_DIR / "test"  / "pajaro", "pajaro_test")

print("Copia de imágenes a la estructura final completada correctamente ✅")


Copia de imágenes a la estructura final completada correctamente ✅


In [46]:
# Celda 12: Resumen de imágenes por split y por clase

def count_images_in_dir(split_dir: Path):
    """
    Cuenta el número de imágenes en cada subcarpeta de clases dentro de 'split_dir'.
    
    Parámetros:
        split_dir: carpeta correspondiente a train, val o test.
    
    Retorna:
        Diccionario {nombre_clase: cantidad_imágenes}.
    """
    counts = {}
    for class_dir in sorted(split_dir.iterdir()):
        if class_dir.is_dir():
            n_imgs = 0
            for ext in IMAGE_EXTENSIONS:
                n_imgs += len(list(class_dir.glob(f"*{ext}")))
            counts[class_dir.name] = n_imgs
    return counts

print("=== Resumen final del dataset de animales preparado ===")
for split in ["train", "val", "test"]:
    split_dir = FINAL_DATA_DIR / split
    counts = count_images_in_dir(split_dir)
    print(f"{split.upper()}:", counts)


=== Resumen final del dataset de animales preparado ===
TRAIN: {'gato': 3507, 'pajaro': 3507, 'perro': 3507}
VAL: {'gato': 751, 'pajaro': 751, 'perro': 751}
TEST: {'gato': 753, 'pajaro': 753, 'perro': 753}
